In [1]:
from pathlib import Path

Path.cwd()

PosixPath('/Users/jenniferhu/IDX-ds-su26')

In [2]:
list(Path(".").glob("*"))

[PosixPath('02_preprocessing.ipynb'),
 PosixPath('.DS_Store'),
 PosixPath('01_exploration.ipynb'),
 PosixPath('residential_single_family_eda.csv'),
 PosixPath('.ipynb_checkpoints'),
 PosixPath('data')]

In [19]:
%pip install scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 35.9 MB/s  0:00:008.5 MB/s eta 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn]━━━━━ 2/3 [scikit-learn]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("residential_single_family_eda.csv")

df.head()

,CloseDate,ClosePrice,LivingArea,Bedrooms,Bathrooms,LotSize,PropertyType,PropertySubType,source_file
0,2024-04-02,250000.0,1723.0,2.0,2.0,8100.0,Residential,SingleFamilyResidence,CRMLSSold202404_filled.csv
1,2024-04-30,413700.0,2285.0,3.0,3.0,43560.0,Residential,SingleFamilyResidence,CRMLSSold202404_filled.csv
2,2024-04-03,725000.0,1716.0,5.0,2.0,10557.0,Residential,SingleFamilyResidence,CRMLSSold202404_filled.csv
3,2024-04-12,600000.0,1600.0,3.0,3.0,8130.0,Residential,SingleFamilyResidence,CRMLSSold202404_filled.csv
4,2024-04-10,1810000.0,2001.0,4.0,2.0,15600.0,Residential,SingleFamilyResidence,CRMLSSold202404_filled.csv


In [21]:
df.shape

(399157, 9)

In [22]:
df.columns.tolist()

['CloseDate',
 'ClosePrice',
 'LivingArea',
 'Bedrooms',
 'Bathrooms',
 'LotSize',
 'PropertyType',
 'PropertySubType',
 'source_file']

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 399157 entries, 0 to 399156
Data columns (total 9 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   CloseDate        399157 non-null  object 
 1   ClosePrice       399155 non-null  float64
 2   LivingArea       398947 non-null  float64
 3   Bedrooms         399157 non-null  float64
 4   Bathrooms        399082 non-null  float64
 5   LotSize          392331 non-null  float64
 6   PropertyType     399157 non-null  object 
 7   PropertySubType  399157 non-null  object 
 8   source_file      399157 non-null  object 
dtypes: float64(5), object(4)
memory usage: 27.4+ MB


In [24]:
df["CloseDate"] = pd.to_datetime(df["CloseDate"], errors="coerce")
df = df.sort_values("CloseDate").reset_index(drop=True)
df["CloseMonth"] = df["CloseDate"].dt.to_period("M")
df[["CloseDate", "CloseMonth"]].head()

,CloseDate,CloseMonth
0,2022-01-01,2022-01
1,2022-01-01,2022-01
2,2022-01-02,2022-01
3,2022-01-03,2022-01
4,2022-01-03,2022-01


In [25]:
df["CloseMonth"].value_counts().sort_index()

CloseMonth
2022-01     2313
2022-02     1227
2022-03      873
2022-04      408
2022-05      229
2022-06      171
2022-07      100
2022-08       72
2022-09       50
2022-10       38
2022-11       23
2022-12       17
2023-01       12
2023-02       19
2023-03       20
2023-04       80
2023-05     1832
2023-06     8601
2023-07     9888
2023-08    11715
2023-09    10744
2023-10    11033
2023-11     9668
2023-12     9518
2024-01     8322
2024-02     9564
2024-03    11634
2024-04    12745
2024-05    13831
2024-06    12545
2024-07    13378
2024-08    12433
2024-09    10909
2024-10    12347
2024-11    10692
2024-12    10624
2025-01     8144
2025-02     8851
2025-03    10610
2025-04    11880
2025-05    11777
2025-06    11701
2025-07    12114
2025-08    11454
2025-09    11456
2025-10    12029
2025-11     9739
2025-12    10455
2026-01     7490
2026-02     8550
2026-03    11177
2026-04    12031
2026-05    12024
Freq: M, Name: count, dtype: int64

In [26]:
target = "ClosePrice"
numeric_features = ["LivingArea", "Bedrooms", "Bathrooms", "LotSize"]
base_cols = ["CloseDate", "CloseMonth", target] + numeric_features
model_df = df[base_cols].copy()

model_df.head()

,CloseDate,CloseMonth,ClosePrice,LivingArea,Bedrooms,Bathrooms,LotSize
0,2022-01-01,2022-01,535000.0,2061.0,4.0,2.0,12232.0
1,2022-01-01,2022-01,560000.0,1546.0,3.0,2.0,27225.0
2,2022-01-02,2022-01,3300000.0,3085.0,3.0,3.0,42237.0
3,2022-01-03,2022-01,1700000.0,2916.0,4.0,3.0,5654.0
4,2022-01-03,2022-01,405000.0,1632.0,3.0,2.0,6151.0


In [27]:
model_df = model_df.dropna(subset=["CloseDate", "ClosePrice"]).copy()

model_df.isna().sum()

CloseDate        0
CloseMonth       0
ClosePrice       0
LivingArea     210
Bedrooms         0
Bathrooms       75
LotSize       6826
dtype: int64

In [28]:
for col in numeric_features:
    model_df[f"{col}_missing"] = model_df[col].isna().astype(int)

missing_flag_cols = [f"{col}_missing" for col in numeric_features]
all_features = numeric_features + missing_flag_cols

model_df.head()

,CloseDate,CloseMonth,ClosePrice,LivingArea,Bedrooms,Bathrooms,LotSize,LivingArea_missing,Bedrooms_missing,Bathrooms_missing,LotSize_missing
0,2022-01-01,2022-01,535000.0,2061.0,4.0,2.0,12232.0,0,0,0,0
1,2022-01-01,2022-01,560000.0,1546.0,3.0,2.0,27225.0,0,0,0,0
2,2022-01-02,2022-01,3300000.0,3085.0,3.0,3.0,42237.0,0,0,0,0
3,2022-01-03,2022-01,1700000.0,2916.0,4.0,3.0,5654.0,0,0,0,0
4,2022-01-03,2022-01,405000.0,1632.0,3.0,2.0,6151.0,0,0,0,0


In [29]:
all_months = sorted(model_df["CloseMonth"].dropna().unique())
all_months

[Period('2022-01', 'M'),
 Period('2022-02', 'M'),
 Period('2022-03', 'M'),
 Period('2022-04', 'M'),
 Period('2022-05', 'M'),
 Period('2022-06', 'M'),
 Period('2022-07', 'M'),
 Period('2022-08', 'M'),
 Period('2022-09', 'M'),
 Period('2022-10', 'M'),
 Period('2022-11', 'M'),
 Period('2022-12', 'M'),
 Period('2023-01', 'M'),
 Period('2023-02', 'M'),
 Period('2023-03', 'M'),
 Period('2023-04', 'M'),
 Period('2023-05', 'M'),
 Period('2023-06', 'M'),
 Period('2023-07', 'M'),
 Period('2023-08', 'M'),
 Period('2023-09', 'M'),
 Period('2023-10', 'M'),
 Period('2023-11', 'M'),
 Period('2023-12', 'M'),
 Period('2024-01', 'M'),
 Period('2024-02', 'M'),
 Period('2024-03', 'M'),
 Period('2024-04', 'M'),
 Period('2024-05', 'M'),
 Period('2024-06', 'M'),
 Period('2024-07', 'M'),
 Period('2024-08', 'M'),
 Period('2024-09', 'M'),
 Period('2024-10', 'M'),
 Period('2024-11', 'M'),
 Period('2024-12', 'M'),
 Period('2025-01', 'M'),
 Period('2025-02', 'M'),
 Period('2025-03', 'M'),
 Period('2025-04', 'M'),


In [30]:
test_month = all_months[-1]
test_month

Period('2026-05', 'M')

In [31]:
test_df = model_df[model_df["CloseMonth"] == test_month].copy()

test_df.shape

(12024, 11)

In [32]:
def make_time_split(data, x_months):
    all_months = sorted(data["CloseMonth"].dropna().unique())
    test_month = all_months[-1]
    train_months = all_months[-(x_months + 1):-1]
    train_df = data[data["CloseMonth"].isin(train_months)].copy()
    test_df = data[data["CloseMonth"] == test_month].copy()
    return train_df, test_df, train_months, test_month

In [33]:
train_df, test_df, train_months, test_month = make_time_split(model_df, x_months=3)

print("Train months:", train_months)
print("Test month:", test_month)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train months: [Period('2026-02', 'M'), Period('2026-03', 'M'), Period('2026-04', 'M')]
Test month: 2026-05
Train shape: (31758, 11)
Test shape: (12024, 11)


In [35]:
numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])

preprocessor = ColumnTransformer(transformers=[("num", numeric_transformer, all_features)])

In [36]:
X_months = 3

train_df, test_df, train_months, test_month = make_time_split(model_df, X_months)
X_train = train_df[all_features]
y_train = train_df[target]
X_test = test_df[all_features]
y_test = test_df[target]

model = Pipeline(steps=[("preprocessor", preprocessor),("model", Ridge())])

model.fit(X_train, y_train)
preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print("X months:", X_months)
print("Train months:", train_months)
print("Test month:", test_month)
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

X months: 3
Train months: [Period('2026-02', 'M'), Period('2026-03', 'M'), Period('2026-04', 'M')]
Test month: 2026-05
MAE: 644601.8142909019
RMSE: 1443807.4096045594
R2: 0.25970725093146185


In [37]:
results = []
max_x = len(all_months) - 1
for x in range(1, max_x + 1):
    train_df, test_df, train_months, test_month = make_time_split(model_df, x)
    X_train = train_df[all_features]
    y_train = train_df[target]
    X_test = test_df[all_features]
    y_test = test_df[target]
    model = Pipeline(steps=[("preprocessor", preprocessor), ("model", Ridge())])
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results.append({
        "X_months": x,
        "train_months": ", ".join(str(m) for m in train_months),
        "test_month": str(test_month),
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results)
results_df.sort_values("MAE")

,X_months,train_months,test_month,train_rows,test_rows,MAE,RMSE,R2
34,35,"2023-06, 2023-07, 2023-08, 2023-09, 2023-10, 2...",2026-05,379647,12024,612195.740774,1.433103e+06,0.270644
33,34,"2023-07, 2023-08, 2023-09, 2023-10, 2023-11, 2...",2026-05,371046,12024,612390.907034,1.433120e+06,0.270626
32,33,"2023-08, 2023-09, 2023-10, 2023-11, 2023-12, 2...",2026-05,361158,12024,612839.468323,1.433227e+06,0.270518
31,32,"2023-09, 2023-10, 2023-11, 2023-12, 2024-01, 2...",2026-05,349443,12024,613129.923361,1.433138e+06,0.270608
35,36,"2023-05, 2023-06, 2023-07, 2023-08, 2023-09, 2...",2026-05,381479,12024,613409.401246,1.435125e+06,0.268584
36,37,"2023-04, 2023-05, 2023-06, 2023-07, 2023-08, 2...",2026-05,381559,12024,613431.806976,1.435123e+06,0.268587
30,31,"2023-10, 2023-11, 2023-12, 2024-01, 2024-02, 2...",2026-05,338699,12024,613533.736389,1.433416e+06,0.270325
37,38,"2023-03, 2023-04, 2023-05, 2023-06, 2023-07, 2...",2026-05,381579,12024,613642.178641,1.435717e+06,0.267981
38,39,"2023-02, 2023-03, 2023-04, 2023-05, 2023-06, 2...",2026-05,381598,12024,613749.511666,1.435717e+06,0.267980
39,40,"2023-01, 2023-02, 2023-03, 2023-04, 2023-05, 2...",2026-05,381610,12024,613867.258641,1.435703e+06,0.267995


In [39]:
cleaned_path = "cleaned_residential_single_family_preprocessed.csv"
model_df.to_csv(cleaned_path, index=False)
cleaned_path

'cleaned_residential_single_family_preprocessed.csv'

## Week 3 Preprocessing Summary

In this notebook, I prepared the filtered residential single-family property dataset for modeling. I started from the cleaned EDA dataset created in Week 2, converted `CloseDate` to a datetime field, and created a `CloseMonth` variable for time-based splitting.

For missing values, I dropped rows missing the target variable `ClosePrice` or the date variable `CloseDate`, since those are required for supervised learning and time-based train/test splitting. For numerical feature columns, I used median imputation inside a preprocessing pipeline. I also created missing-value indicator columns so that the model can capture whether missingness itself is informative.

The target variable is `ClosePrice`. The initial feature set includes `LivingArea`, `Bedrooms`, `Bathrooms`, and `LotSize`, along with missing-value flags for these variables.

To follow the requirements for this week, I used the most recent month of available data as the test set. I then treated the number of immediately preceding months used for training as a tunable parameter `X`. I experimented with different values of `X` and compared model performance using MAE, RMSE, and R².

Finally, I saved the cleaned modeling dataset as `cleaned_residential_single_family_preprocessed.csv`.